# MinbarAI remote translation endpoint (Colab / Kaggle)

Serves **translategemma:12b** on the free GPU and exposes it through a Cloudflare quick tunnel.

**Setup**
- Colab: `Runtime > Change runtime type > T4 GPU`
- Kaggle: `Settings > Accelerator > GPU T4 x2` **and** `Settings > Internet > On`

**Run all cells.** The tunnel cell prints a line like:
```
REMOTE_OLLAMA_HOST=https://xxxx-xxxx.trycloudflare.com
```
Copy it into `.env` at the MinbarAI project root, then start the app. The app checks the endpoint every 30 s and falls back to local `translategemma:4b` automatically when the session dies.

Keep this notebook tab open during the khutbah (last cell keeps the session alive). Pre-warm ~15 min before start: the 12B pull takes a while.

In [ ]:
# 1. Install Ollama (installer needs zstd; missing on Kaggle images)
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Start the Ollama server and pull the model (~8 GB, takes a few minutes)
import os, subprocess, time

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"   # listen on all interfaces for the tunnel
env["OLLAMA_ORIGINS"] = "*"            # accept requests coming through the tunnel hostname
env["OLLAMA_KEEP_ALIVE"] = "-1"        # never unload the model between chunks
server = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)

!ollama pull translategemma:12b

In [ ]:
# 3. Warm up + sanity check on the GPU
import json, time, urllib.request

t0 = time.time()
req = urllib.request.Request(
    "http://localhost:11434/api/chat",
    data=json.dumps({
        "model": "translategemma:12b",
        "stream": False,
        "messages": [{"role": "user", "content": (
            "You are a professional Arabic (ar) to German (de-DE) translator. "
            "Produce only the German translation. "
            "Please translate the following Arabic text into German:\n\n\n"
            "\u0627\u0644\u062d\u0645\u062f \u0644\u0644\u0647 \u0631\u0628 \u0627\u0644\u0639\u0627\u0644\u0645\u064a\u0646"
        )}],
        "options": {"temperature": 0}
    }).encode(),
    headers={"Content-Type": "application/json"},
)
resp = json.loads(urllib.request.urlopen(req, timeout=300).read())
print(f"{time.time()-t0:.1f}s ->", resp["message"]["content"])

In [ ]:
# 4. Open the tunnel — copy the printed REMOTE_OLLAMA_HOST line into MinbarAI/.env
import re, subprocess

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n" + "=" * 60)
        print("REMOTE_OLLAMA_HOST=" + m.group(0))
        print("=" * 60)
        break

In [ ]:
# 5. Keep-alive — leave this running during the khutbah
import time

while True:
    time.sleep(600)
    print("alive", time.strftime("%H:%M:%S"))